In [1]:
import json
from pathlib import Path

import joblib

In [3]:
docs = []
with open('data/pages.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        docs.append(json.loads(line))

print(len(docs))
docs[0]

63


{'url': 'https://kandydacipb.edu.pl/rekrutacja/',
 'title': 'Rekrutacja - Politechnika Białostocka',
 'text': 'Rekrutacja - Politechnika Białostocka Facebook Instagram youtube linkedin tiktok \uf002 Wyszukaj kierunek \ue0e7 Oblicz wzór rekrutacyjny \uf501 Rekrutacja I stopnia Sprawdź kierunki studiów. Przygotowaliśmy je z myślą o tobie \uf501 Rekrutacja II stopnia Czekają na ciebie interesujące studia magisterskie \uf549 Studia podyplomowe Wejdź na wyższy poziom edukacji! Poznaj ofertę dla specjalistów \uf19d Szkoła doktorska Sięgnij po więcej wiedzy. Zdobądź stopień doktora w jednej z siedmiu dyscyplin \uf7a2 Cudzoziemcy You’re warmly welcome! \uf70e Potwierdzenie efektów uczenia się Aplikuj online aplikuj online << Starsze wpisy',
 'category': 'rekrutacja'}

In [4]:
NOISE = [
    "Facebook Instagram youtube linkedin tiktok",
    "Strona główna",
    "Aplikuj online",
]

def clean_doc_text(text):
    for n in NOISE:
        text = text.replace(n, " ")
    return " ".join(text.split())

def chunk_text(text, chunk_size=1200, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

def build_chunks(docs):
    chunks = []

    for doc in docs:
        text = clean_doc_text(doc.get("text", ""))

        for i, chunk in enumerate(chunk_text(text)):
            chunks.append({
                "url": doc.get("url", ""),
                "title": doc.get("title", ""),
                "category": doc.get("category", ""),
                "chunk_id": i,
                "text": chunk,
            })

    return chunks

In [5]:
chunks = build_chunks(docs)
texts = [chunk["text"] for chunk in chunks]

print(len(chunks))
chunks[0]

171


{'url': 'https://kandydacipb.edu.pl/rekrutacja/',
 'title': 'Rekrutacja - Politechnika Białostocka',
 'category': 'rekrutacja',
 'chunk_id': 0,
 'text': 'Rekrutacja - Politechnika Białostocka \uf002 Wyszukaj kierunek \ue0e7 Oblicz wzór rekrutacyjny \uf501 Rekrutacja I stopnia Sprawdź kierunki studiów. Przygotowaliśmy je z myślą o tobie \uf501 Rekrutacja II stopnia Czekają na ciebie interesujące studia magisterskie \uf549 Studia podyplomowe Wejdź na wyższy poziom edukacji! Poznaj ofertę dla specjalistów \uf19d Szkoła doktorska Sięgnij po więcej wiedzy. Zdobądź stopień doktora w jednej z siedmiu dyscyplin \uf7a2 Cudzoziemcy You’re warmly welcome! \uf70e Potwierdzenie efektów uczenia się aplikuj online << Starsze wpisy'}

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=30000,
    min_df=1,
    sublinear_tf=True,
)

X = vectorizer.fit_transform(texts)

print(X.shape)

(171, 13147)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query, k=5):
    q = vectorizer.transform([query])

    scores = cosine_similarity(q, X).flatten()

    best = scores.argsort()[::-1][:k]

    for idx in best:
        print("=" * 50)
        print("TITLE:", chunks[idx]["title"])
        print("URL:", chunks[idx]["url"])
        print("CATEGORY:", chunks[idx]["category"])
        print("CHUNK:", chunks[idx]["chunk_id"])
        print("SCORE:", scores[idx])
        print(chunks[idx]["text"][:500])

In [8]:
search("jakie są progi na informatykę")

TITLE: Progi punktowe - Politechnika Białostocka
URL: https://kandydacipb.edu.pl/studia-i-stopnia/progi-punktowe
CATEGORY: rekrutacja
CHUNK: 0
SCORE: 0.1435255776139425
Progi punktowe - Politechnika Białostocka 9 Studia I stopnia 9 Progi punktowe Progi punktowe Progi punktowe obowiązujące w rekrutacji na studia stacjonarne I stopnia na semestr zimowy w roku akademickim 2025/2026. Wydział Informatyki: Data Science: 128 informatyka: 128 matematyka stosowana: 128 Wydział Inżynierii Zarządzania: logistyka: 50 turystyka i rekreacja: 60 zarządzanie: 60 zarządzanie finansami i rachunkowość: 50 zarządzanie i inżynieria produkcji: 50 zarządzanie projektami: 50 Wydział M
TITLE: Faq - Politechnika Białostocka
URL: https://kandydacipb.edu.pl/faq
CATEGORY: rekrutacja
CHUNK: 6
SCORE: 0.05803457800718347
nia w systemie IRK? Podanie dostępne będzie w systemie dopiero po uzupełnieniu wszystkich wymaganych danych i po przesłaniu do systemu IRK zdjęcia. Wyjeżdżam i nie będę mógł złożyć dokumentów osobiśc

In [9]:
Path("data").mkdir(exist_ok=True)

index_data = {
    "vectorizer": vectorizer,
    "matrix": X,
    "chunks": chunks,
}

joblib.dump(index_data, "data/tfidf_index.joblib")
print("Zapisano indeks TF-IDF: data/tfidf_index.joblib")

Zapisano indeks TF-IDF: data/tfidf_index.joblib
